# 04 — Review Artifact Reader

Use this notebook to review one row without opening many files manually. It is the clean reviewer gateway for the concise row packet.

```text
output/<row_id>/
├── final_row.csv
├── review_summary.md
├── review_decision.json
├── candidate_decisions.csv
└── product_coding_input.json
```

```mermaid
flowchart TD
    A[Set ROW_ARTIFACT_DIR] --> B[Load review_decision.json]
    B --> C[Show selected URL]
    C --> D[Show gate checks]
    D --> E[Show why selected]
    E --> F[Show candidate decisions]
    F --> G[Reviewer action]
```


In [ ]:
from pathlib import Path
import json

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROJECT_ROOT


## 1. Set row artifact path

Point this to one row folder generated by Notebook 01 or Notebook 02.


In [ ]:
ROW_ARTIFACT_DIR = PROJECT_ROOT / "output" / "PUT_ROW_ID_HERE"

review_json = ROW_ARTIFACT_DIR / "review_decision.json"
candidate_csv = ROW_ARTIFACT_DIR / "candidate_decisions.csv"
summary_md = ROW_ARTIFACT_DIR / "review_summary.md"
coding_json = ROW_ARTIFACT_DIR / "product_coding_input.json"

required = [review_json, candidate_csv, summary_md, coding_json]
status = pd.DataFrame({
    "artifact": [p.name for p in required],
    "path": [str(p) for p in required],
    "exists": [p.exists() for p in required],
})
display(status)

missing = [p for p in required if not p.exists()]
assert not missing, "Missing required concise review artifacts. Check ROW_ARTIFACT_DIR."


In [ ]:
payload = json.loads(review_json.read_text(encoding="utf-8"))
candidates = pd.read_csv(candidate_csv, dtype=str)

decision = payload.get("decision", {})
checks = payload.get("checks", {})
decision


## 2. Decision snapshot


In [ ]:
pd.DataFrame([decision]).T.rename(columns={0: "value"})


## 3. Gate checks


In [ ]:
pd.DataFrame([checks]).T.rename(columns={0: "value"})


## 4. Why selected


In [ ]:
for point in payload.get("why_selected", []):
    print("-", point)


## 5. How decided


In [ ]:
for point in payload.get("how_decided", []):
    print("-", point)


## 6. AI/model summary

This shows observable model/planner outputs only. It is not hidden chain-of-thought.


In [ ]:
for point in payload.get("ai_reasoning_summary", []):
    print("-", point)


## 7. Candidate decisions


In [ ]:
review_columns = [
    "rank", "selected", "decision", "url", "confidence", "validation_status",
    "identity_status", "exact_product_check", "country_check", "retailer_check",
    "scrapable", "product_page", "reason",
]
display(candidates[[c for c in review_columns if c in candidates.columns]])


## 8. Human-readable summary


In [ ]:
print(summary_md)
print("\nFirst 1200 characters:\n")
print(summary_md.read_text(encoding="utf-8")[:1200])


## 9. Product coding input


In [ ]:
coding_payload = json.loads(coding_json.read_text(encoding="utf-8"))
pd.DataFrame([coding_payload]).T.rename(columns={0: "value"}).head(60)
